In [2]:
import tensorflow as tf
import numpy as np

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

TensorFlow version: 2.21.0
NumPy version: 2.2.6


In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os

print(f"TensorFlow Version: {tf.__version__}")
# This will confirm if your GPU is being recognized for faster training
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow Version: 2.21.0
Num GPUs Available:  0


In [4]:
# Define your base directories
base_dir = 'dataset/Data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# MobileNetV2 expects 224x224 images
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Loading Training Data:")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("\nLoading Validation Data:")
val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    shuffle=True,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names

Loading Training Data:
Found 43430 files belonging to 38 classes.

Loading Validation Data:
Found 5417 files belonging to 38 classes.


In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [6]:
# Data augmentation to prevent overfitting
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
])

# MobileNetV2 expects pixel values mapped from [0, 255] to [-1, 1]
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# Create the base model from the pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,),
                                               include_top=False,
                                               weights='imagenet')

# Freeze the base model
base_model.trainable = False

# Build the final architecture
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x) # Dropout helps prevent overfitting
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,306,662 (8.80 MB)

 Trainable params: 48,678 (190.15 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
base_learning_rate = 0.001
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

# Stop training when validation loss stops improving for 3 epochs
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

initial_epochs = 15

print("Starting training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=initial_epochs,
    callbacks=[early_stopping]
)

Starting training...
Epoch 1/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 226s 158ms/step - accuracy: 0.8232 - loss: 0.6148 - val_accuracy: 0.9090 - val_loss: 0.3180
Epoch 2/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 242s 178ms/step - accuracy: 0.9024 - loss: 0.3104 - val_accuracy: 0.9132 - val_loss: 0.2800
Epoch 3/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 240s 177ms/step - accuracy: 0.9126 - loss: 0.2719 - val_accuracy: 0.9251 - val_loss: 0.2487
Epoch 4/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 244s 180ms/step - accuracy: 0.9178 - loss: 0.2538 - val_accuracy: 0.9348 - val_loss: 0.2077
Epoch 5/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 256s 189ms/step - accuracy: 0.9220 - loss: 0.2417 - val_accuracy: 0.9321 - val_loss: 0.2219
Epoch 6/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 247s 182ms/step - accuracy: 0.9238 - loss: 0.2318 - val_accuracy: 0.9262 - val_loss: 0.2295
Epoch 7/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 234s 172ms/step - accuracy: 0.9271 - loss: 0.2254 - val_accuracy: 0.9284 - val_loss: 0.2186


In [8]:
print("Evaluating on Test Data:")
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

loss, accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {accuracy*100:.2f}%')

# Save the model in the Keras format
model.save('plant_disease_model.keras')
print("Model successfully saved as 'plant_disease_model.keras'!")

Evaluating on Test Data:
Found 5459 files belonging to 38 classes.
171/171 ━━━━━━━━━━━━━━━━━━━━ 21s 122ms/step - accuracy: 0.9339 - loss: 0.2061
Test accuracy: 93.39%
Model successfully saved as 'plant_disease_model.keras'!
